<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L03-reranking-measured/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L03-reranking-measured/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T06-L03-reranking-measured/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T06-L03-reranking-measured/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T06-L03 · Reranking, and what it costs

**You will build:** two learned rerankers — logistic regression trained pointwise, and a
RankNet-style pairwise reranker — over features a first-stage hybrid retriever already
computed, plus the discipline that keeps a reported gain honest: choose by validation, touch
the test split exactly once.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track, T06-L01-retrieval-from-scratch,
T06-L02-chunking-and-context-budget.

T06-L01 built a first-stage retriever tuned for RECALL: cast a wide net, cheaply, per
document. This lesson builds the second stage — a reranker that looks harder at that first
stage's own short list, for PRECISION — and then measures, rather than assumes, three things
every reranking project needs to know: how much it actually gained, how deep into the
candidate list it is worth reordering, and whether the number you are about to report was
earned honestly or by peeking at the split you were supposed to hold out.

By the end you will be able to:

1. Implement query-term coverage and term-proximity features over a first-stage candidate
   pool, and combine them with the candidates' own BM25 score, dense cosine and first-stage
   ranks into one feature matrix.
2. Implement a pointwise logistic-regression reranker and a pairwise RankNet-style reranker,
   both trained by full-batch gradient descent from numpy alone.
3. Measure nDCG@10 and MRR gains of each reranker over the first-stage ranking, with paired
   bootstrap confidence intervals over queries.
4. Measure the rerank-depth curve — quality against how many first-stage candidates are
   reranked — and the wall-clock cost that buys, printed with its unit.
5. Detect, with a tripwire the notebook prints rather than a written assertion, whether a
   model-selection step read the held-out test queries, and measure how much a reranker
   chosen by peeking at test overstates its gain relative to queries never touched during
   selection.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import math
import re
import sys
import time
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260923          # this lesson's one fixed seed: the corpus, the query splits, exercise
                          # 6's simulated "recency" tags and every bootstrap resample all derive
                          # from it, so re-running prints the same results on any machine.
EVAL_K = 10                # the cutoff used for nDCG@k throughout this lesson.
N_BOOT = 2000             # bootstrap resamples for every confidence interval below.
NOISE_SEED = SEED + 2     # exercise 6's demo only: seeds its simulated "recency" tags (SEED + 1
                          # seeds the query splits). Derived from SEED, never searched for.
N_TRIALS = 8              # exercise 6's demo: independent repeats of the whole model selection.
N_NOISE_MODELS = 4        # exercise 6's demo: rival "recency" models per trial, besides plain.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("query_term_coverage", "term_proximity"),
    "exercise 2": ("build_feature_matrix",),
    "exercise 3": ("sigmoid", "train_pointwise_logreg"),
    "exercise 4": ("train_pairwise_ranknet",),
    "exercise 5": ("rerank",),
    "exercise 6": ("select_and_report",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (bm25_rank)"; several -> "exercises 3, 6 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 1. What T06-L01 and T06-L02 already gave you (given, unchanged)

A two-stage retrieval system splits the work: a first stage casts a wide net — high recall,
cheap per document — and a second stage, this lesson, looks harder at a short list of
candidates the first stage already found. That short list is this lesson's whole world: every
exercise below reorders a first-stage candidate pool, never the whole corpus again.

The first stage here is T06-L01's hybrid retriever — BM25 fused with a from-scratch dense
retriever by reciprocal rank fusion — carried forward with its mechanism unchanged (only the
number each retriever contributes, `POOL_N`, is this lesson's own), because a reranker's
features are the numbers that retriever already computed on the way to its candidate list:
a BM25 score, a dense cosine, and each one's rank. Two small changes make every result below
print the same on every machine, and a note beside each says why: `dense_rank` rounds its
cosines to 6 decimals, and the bootstrap sums each resample exactly. Read this section;
nothing in it is graded.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def tokenize(text: str) -> list[str]:
    """Lower-case `text` and split it into tokens (T06-L01, unchanged)."""
    return _TOKEN_RE.findall(text.lower())


class Index(NamedTuple):
    doc_term_counts: list[dict[str, int]]
    doc_lengths: list[int]
    avgdl: float
    df: dict[str, int]
    n_docs: int


def build_index(docs: Sequence[str]) -> Index:
    """Tokenise every document in `docs` and index it for BM25 (T06-L01, unchanged)."""
    doc_term_counts: list[dict[str, int]] = []
    doc_lengths: list[int] = []
    df: dict[str, int] = {}
    for doc in docs:
        counts: dict[str, int] = {}
        for token in tokenize(doc):
            counts[token] = counts.get(token, 0) + 1
        doc_term_counts.append(counts)
        doc_lengths.append(sum(counts.values()))
        for term in counts:
            df[term] = df.get(term, 0) + 1
    n_docs = len(docs)
    avgdl = sum(doc_lengths) / n_docs if n_docs else 0.0
    return Index(doc_term_counts, doc_lengths, avgdl, df, n_docs)


def bm25_idf(term: str, index: Index) -> float:
    """BM25's inverse-document-frequency weight for `term` (T06-L01, unchanged)."""
    n_t = index.df.get(term, 0)
    N = index.n_docs
    return math.log((N - n_t + 0.5) / (n_t + 0.5))


def bm25_score(query_tokens: Sequence[str], doc_id: int, index: Index,
               k1: float = 1.2, b: float = 0.75) -> float:
    """BM25's score for document `doc_id` against `query_tokens` (T06-L01, unchanged)."""
    query_term_counts: dict[str, int] = {}
    for token in query_tokens:
        query_term_counts[token] = query_term_counts.get(token, 0) + 1
    counts = index.doc_term_counts[doc_id]
    dl = index.doc_lengths[doc_id]
    avgdl = index.avgdl if index.avgdl else 1.0
    score = 0.0
    for term, qtf in query_term_counts.items():
        tf = counts.get(term, 0)
        if tf == 0:
            continue
        idf = bm25_idf(term, index)
        denom = k1 * ((1 - b) + b * dl / avgdl) + tf
        score += qtf * idf * tf / denom
    return score


def bm25_rank(query_tokens: Sequence[str], index: Index,
              k1: float = 1.2, b: float = 0.75) -> list[tuple[int, float]]:
    """Rank every document in `index` against `query_tokens` by BM25 score (T06-L01, unchanged)."""
    scored = [(doc_id, bm25_score(query_tokens, doc_id, index, k1, b))
              for doc_id in range(index.n_docs)]
    scored = [(doc_id, score) for doc_id, score in scored if score != 0.0]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    return scored


def tfidf_idf(term: str, index: Index) -> float:
    """A smoothed idf for the TF-IDF vector space (T06-L01, unchanged): log(N/n_t) + 1."""
    n_t = index.df.get(term, 0)
    if n_t == 0:
        return 0.0
    return math.log(index.n_docs / n_t) + 1.0


def tfidf_matrix(index: Index) -> tuple[np.ndarray, list[str], dict[str, int], np.ndarray]:
    """The corpus's TF-IDF matrix, unit-row-normalised (T06-L01, unchanged)."""
    vocab = sorted(index.df)
    vocab_pos = {term: i for i, term in enumerate(vocab)}
    idf_weights = np.array([tfidf_idf(term, index) for term in vocab])
    M = np.zeros((index.n_docs, len(vocab)))
    for doc_id, counts in enumerate(index.doc_term_counts):
        for term, tf in counts.items():
            j = vocab_pos[term]
            M[doc_id, j] = (1.0 + math.log(tf)) * idf_weights[j]
    norms = np.linalg.norm(M, axis=1)
    norms_safe = np.where(norms == 0, 1.0, norms)
    M = M / norms_safe[:, None]
    return M, vocab, vocab_pos, idf_weights


def truncated_svd(matrix: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """The rank-`k` truncated SVD of `matrix`, clamped to the available rank (T06-L01, unchanged)."""
    U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
    k = min(k, S.shape[0])
    return U[:, :k], S[:k], Vt[:k, :]


class DenseIndex(NamedTuple):
    vocab_pos: dict[str, int]
    idf_weights: np.ndarray
    Vt_k: np.ndarray
    doc_embeddings: np.ndarray


def build_dense_index(index: Index, k: int) -> DenseIndex:
    """A corpus indexed for dense retrieval, reduced to `k` latent dimensions (T06-L01, unchanged)."""
    M, vocab, vocab_pos, idf_weights = tfidf_matrix(index)
    U_k, S_k, Vt_k = truncated_svd(M, k)
    projected = M @ Vt_k.T
    norms = np.linalg.norm(projected, axis=1)
    norms_safe = np.where(norms == 0, 1.0, norms)
    projected = projected / norms_safe[:, None]
    return DenseIndex(vocab_pos, idf_weights, Vt_k, projected)


def embed_query(query_tokens: Sequence[str], dense_index: DenseIndex) -> np.ndarray:
    """Project `query_tokens` into `dense_index`'s reduced space, unit length (T06-L01, unchanged)."""
    counts: dict[str, int] = {}
    for token in query_tokens:
        counts[token] = counts.get(token, 0) + 1
    vec = np.zeros(dense_index.Vt_k.shape[1])
    for term, tf in counts.items():
        j = dense_index.vocab_pos.get(term)
        if j is None:
            continue
        vec[j] = (1.0 + math.log(tf)) * dense_index.idf_weights[j]
    projected = dense_index.Vt_k @ vec
    norm = np.linalg.norm(projected)
    if norm == 0:
        return projected
    return projected / norm


# The one change from T06-L01's dense_rank: every cosine is rounded to `_DENSE_TIE_DECIMALS`
# decimals before ranking. Two of this corpus's near-misses are mirror images — same topic, same
# filler words, each missing a different one of two core words that play exactly the same part
# in every other document of that topic — so their dense embeddings are mathematically the same
# vector and they tie against every query. In floating point one of the two wins by its last
# bit, and which one depends on the linear-algebra library doing the arithmetic: it differed
# between macOS and Linux, and moved section 8's numbers. Rounding makes the tie exact again,
# and an exact tie breaks by doc id, as every tie in this track does. Six decimals is far
# coarser than floating-point noise and finer than any real gap between two documents that can
# reach a candidate pool here. Exercise 5 must still never round: how precise a score is gets
# decided where the score is computed, as here; a sort then orders exactly the numbers it is
# handed, as this one does.
_DENSE_TIE_DECIMALS = 6


def dense_rank(query_tokens: Sequence[str], dense_index: DenseIndex) -> list[tuple[int, float]]:
    """Rank every document by cosine similarity to `query_tokens` (T06-L01's, with each cosine
    rounded to `_DENSE_TIE_DECIMALS` decimals first — see the note above)."""
    query_vec = embed_query(query_tokens, dense_index)
    if not np.any(query_vec):
        scores = np.zeros(dense_index.doc_embeddings.shape[0])
    else:
        scores = dense_index.doc_embeddings @ query_vec
    scores = np.round(scores, _DENSE_TIE_DECIMALS)
    order = sorted(range(len(scores)), key=lambda doc_id: (-scores[doc_id], doc_id))
    return [(doc_id, float(scores[doc_id])) for doc_id in order]


def reciprocal_rank_fusion(rankings: Sequence[Sequence[int]],
                            k: int = 60) -> list[tuple[int, float]]:
    """Fuse several ranked document-id lists into one by RRF with constant `k` (T06-L01, unchanged)."""
    totals: dict[int, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            totals[doc_id] = totals.get(doc_id, 0.0) + 1.0 / (k + rank)
    items = list(totals.items())
    items.sort(key=lambda pair: (-pair[1], pair[0]))
    return items


def dcg_at_k(ranked_ids: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """Discounted cumulative gain of `ranked_ids` against `k`, using `relevance` (T06-L01, unchanged)."""
    total = 0.0
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        grade = relevance.get(doc_id, 0)
        if grade:
            total += grade / math.log2(rank + 1)
    return total


def recall_at_k(ranked_ids: Sequence[int], relevant_ids: Sequence[int] | frozenset[int],
                 k: int) -> float:
    """The fraction of `relevant_ids` in the top `k` of `ranked_ids` (T06-L01, unchanged)."""
    relevant = set(relevant_ids)
    if not relevant:
        raise ValueError("recall_at_k needs at least one relevant id")
    top_k = set(ranked_ids[:k])
    return len(top_k & relevant) / len(relevant)


def reciprocal_rank(ranked_ids: Sequence[int],
                     relevant_ids: Sequence[int] | frozenset[int]) -> float:
    """1 / (rank of the first relevant id), or 0.0 if none is present (T06-L01, unchanged)."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked_ids: Sequence[int], relevance: Mapping[int, int], k: int) -> float:
    """Normalised DCG@k of `ranked_ids` against `relevance` (T06-L01, unchanged)."""
    dcg = dcg_at_k(ranked_ids, relevance, k)
    ideal_order = [doc_id for doc_id, _ in sorted(relevance.items(), key=lambda p: -p[1])]
    idcg = dcg_at_k(ideal_order, relevance, k)
    return dcg / idcg if idcg else 0.0


class EvalResult(NamedTuple):
    recall_at_k: list[float]
    reciprocal_rank: list[float]
    ndcg_at_k: list[float]


def evaluate_retriever(rank_fn: Callable[[Sequence[str]], list[tuple[int, float]]],
                        queries: Sequence["Query"], k: int) -> EvalResult:
    """Run `rank_fn` over every query and collect its per-query metrics (T06-L01, unchanged)."""
    recalls, rrs, ndcgs = [], [], []
    for q in queries:
        ranked_ids = [doc_id for doc_id, _ in rank_fn(q.tokens)]
        recalls.append(recall_at_k(ranked_ids, q.relevant_ids, k))
        rrs.append(reciprocal_rank(ranked_ids, q.relevant_ids))
        ndcgs.append(ndcg_at_k(ranked_ids, q.relevance, k))
    return EvalResult(recalls, rrs, ndcgs)


def _exact_means(rows: np.ndarray) -> np.ndarray:
    """Each row's mean, summed with `math.fsum` — the one change from T06-L01's bootstrap. A
    floating-point sum rounds after every addition, so its last bit depends on the ORDER of the
    additions, which is numpy's choice, not this lesson's; `math.fsum` is exact, so it cannot.
    That last bit matters here: one of section 8's interval ends falls exactly halfway between
    two values it could print, so the order of a sum decided which one it printed."""
    return np.array([math.fsum(row) for row in rows]) / rows.shape[1]


def bootstrap_ci(values: Sequence[float], n_boot: int = N_BOOT, seed: int = 0,
                  alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap confidence interval for the mean of `values` (T06-L01's, with
    every mean summed exactly by `_exact_means`)."""
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = _exact_means(arr[resample_idx])
    lower, upper = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(_exact_means(arr[None, :])[0]), float(lower), float(upper)


def paired_bootstrap_diff(a: Sequence[float], b: Sequence[float], n_boot: int = N_BOOT,
                           seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap CI for the mean of `b[i]-a[i]`, paired by query (T06-L01's, with
    every mean summed exactly by `_exact_means`)."""
    a_arr = np.asarray(a, dtype=float)
    b_arr = np.asarray(b, dtype=float)
    n = len(a_arr)
    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    diffs = b_arr[resample_idx] - a_arr[resample_idx]
    boot_means = _exact_means(diffs)
    lower, upper = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(_exact_means((b_arr - a_arr)[None, :])[0]), float(lower), float(upper)


DENSE_K = 30      # latent dimensions for the dense retriever.
POOL_N = 20       # first-stage candidates kept from EACH of BM25 and dense, before union.
RRF_K = 60        # Cormack, Clarke & Buettcher's own default (see claims.yaml).


def hybrid_rank(query_tokens: Sequence[str], index: Index, dense_index: DenseIndex,
                pool_n: int = POOL_N, rrf_k: int = RRF_K) -> list[tuple[int, float]]:
    """T06-L01's hybrid retriever: fuse BM25's and dense's top `pool_n` by RRF (given, unchanged
    mechanism). This IS the first stage — "tuned for recall" by keeping `pool_n` generous."""
    bm25_ids = [doc_id for doc_id, _ in bm25_rank(query_tokens, index)[:pool_n]]
    dense_ids = [doc_id for doc_id, _ in dense_rank(query_tokens, dense_index)[:pool_n]]
    return reciprocal_rank_fusion([bm25_ids, dense_ids], k=rrf_k)


def first_stage_candidates(query_tokens: Sequence[str], index: Index, dense_index: DenseIndex,
                            pool_n: int = POOL_N) -> list[int]:
    """The first stage's CANDIDATE SET: the union of BM25's and dense's own top `pool_n` ids,
    sorted ascending (given — a thin composition, like hybrid_rank above)."""
    bm25_ids = {doc_id for doc_id, _ in bm25_rank(query_tokens, index)[:pool_n]}
    dense_ids = {doc_id for doc_id, _ in dense_rank(query_tokens, dense_index)[:pool_n]}
    return sorted(bm25_ids | dense_ids)

## 2. This lesson's own corpus and query splits (given)

20 topics, each with 4 "core" words. Topic `t`'s **target** document carries a unique record
code plus all 4 core words; four **near-miss** documents each carry only 3 of the 4 (a
different one missing each time) plus random filler — usually shorter, on-topic, NOT the answer.
Four queries per topic, each two ADJACENT core words (cyclically: 0-1, 1-2, 2-3, 3-0) — every
one matches the target AND at least one near-miss on both words, so BM25's length
normalisation alone often prefers the shorter near-miss: read what the next cell prints before
you trust that BM25's own top result is the target. Every TOPIC — all 4 of its queries — is
assigned whole to exactly one of four splits (10/4/3/3 topics: train/val/test/fresh), so no
reranker ever trains, or is chosen, on a document it will later be scored against.

In [ ]:
_TOPIC_WORD_POOL = [
    "alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta", "theta", "iota", "kappa",
    "lambda", "mu", "nu", "xi", "omicron", "pi", "rho", "sigma", "tau", "upsilon",
    "phi", "chi", "psi", "omega", "north", "south", "east", "west", "upper", "lower",
    "inner", "outer", "fore", "aft", "port", "starboard", "apex", "base", "core", "shell",
    "prime", "dual", "triad", "quad", "penta", "hexad", "hepta", "octa", "nona", "deca",
    "azure", "crimson", "amber", "violet", "indigo", "coral", "cobalt", "ochre", "sable",
    "ivory", "willow", "cedar", "maple", "birch", "alder", "rowan", "hazel", "aspen", "elm",
    "fir", "granite", "basalt", "quartz", "slate", "marble", "pumice", "shale", "chalk",
    "flint", "obsidian", "falcon", "osprey", "heron", "kestrel", "merlin", "harrier",
    "plover", "tern", "gull", "wren",
]
_FILLER_POOL = ["report", "today", "please", "status", "system", "regarding", "update",
                "ticket"]
_GENERIC_NOISE_WORDS = ["meeting", "calendar", "printer", "coffee", "parking", "elevator",
    "cafeteria", "badge", "laptop", "monitor", "keyboard", "mouse", "chair", "desk", "window",
    "hallway", "stairwell", "lobby", "reception", "mailroom", "supply", "closet", "whiteboard",
    "projector"]
N_TOPICS = 20
N_NOISE_DOCS = 40
TOPIC_SPLIT_COUNTS = {"train": 10, "val": 4, "test": 3, "fresh": 3}   # topics, not queries


def _topic_words(n_topics: int, seed: int) -> list[list[str]]:
    rng = np.random.default_rng(seed)
    assert n_topics * 4 <= len(_TOPIC_WORD_POOL)
    words = rng.permutation(_TOPIC_WORD_POOL)[: n_topics * 4]
    return [list(words[i * 4:(i + 1) * 4]) for i in range(n_topics)]


def build_corpus(seed: int = SEED) -> tuple[list[str], list[tuple[int, str]]]:
    """Generate this lesson's corpus. Returns `(docs, doc_meta)`: `doc_meta[i]` is
    `(topic_index, role)`, role one of "target", "distractor0".."distractor3", "noise"
    (topic_index -1 for noise)."""
    rng = np.random.default_rng(seed)
    topics = _topic_words(N_TOPICS, seed)
    docs: list[str] = []
    doc_meta: list[tuple[int, str]] = []
    for ti, words in enumerate(topics):
        code = f"rec-{7000 + ti * 13}"
        f1, f2 = rng.choice(_FILLER_POOL, size=2, replace=False)
        extra = rng.choice(_GENERIC_NOISE_WORDS, size=int(rng.integers(0, 4)),
                            replace=False).tolist()
        docs.append(f"{code} {words[0]} {words[1]} {words[2]} {words[3]} {f1} {f2} "
                    + " ".join(extra))
        doc_meta.append((ti, "target"))
        for missing in range(4):
            sub = [w for j, w in enumerate(words) if j != missing]
            f1, f2 = rng.choice(_FILLER_POOL, size=2, replace=False)
            extra = rng.choice(_GENERIC_NOISE_WORDS, size=int(rng.integers(0, 4)),
                                replace=False).tolist()
            docs.append(f"{sub[0]} {sub[1]} {sub[2]} {f1} {f2} " + " ".join(extra))
            doc_meta.append((ti, f"distractor{missing}"))
    for _ in range(N_NOISE_DOCS):
        words = rng.choice(_GENERIC_NOISE_WORDS, size=5, replace=False)
        docs.append("note " + " ".join(words))
        doc_meta.append((-1, "noise"))
    order = rng.permutation(len(docs))
    docs = [docs[i] for i in order]
    doc_meta = [doc_meta[i] for i in order]
    return docs, doc_meta


class Query(NamedTuple):
    """This lesson's own query record — T06-L01's `Query` shape, with `kind` (exact/paraphrase,
    which does not apply here) replaced by `split`, the tag every query in THIS lesson carries."""

    qid: str
    tokens: list[str]
    split: str                     # "train" | "val" | "test" | "fresh"
    relevant_ids: frozenset[int]
    relevance: dict[int, int]


def _topic_splits(n_topics: int, seed: int) -> dict[int, str]:
    rng = np.random.default_rng(seed + 1)   # a different stream from the corpus's own
    order = rng.permutation(n_topics).tolist()
    out: dict[int, str] = {}
    i = 0
    for split, count in TOPIC_SPLIT_COUNTS.items():
        for topic_index in order[i:i + count]:
            out[int(topic_index)] = split
        i += count
    return out


def build_queries(doc_meta: Sequence[tuple[int, str]], seed: int = SEED) -> list["Query"]:
    """Four queries per topic (see section 2), each topic's queries all in the same split."""
    topics = _topic_words(N_TOPICS, seed)
    by_topic: dict[int, dict[str, int]] = {}
    for doc_id, (ti, role) in enumerate(doc_meta):
        if ti == -1:
            continue
        by_topic.setdefault(ti, {})[role] = doc_id
    topic_split = _topic_splits(N_TOPICS, seed)
    adjacent_pairs = [(0, 1), (1, 2), (2, 3), (3, 0)]
    queries: list[Query] = []
    for ti, words in enumerate(topics):
        target_id = by_topic[ti]["target"]
        split = topic_split[ti]
        for pi, (a, b) in enumerate(adjacent_pairs):
            queries.append(Query(f"T{ti}-{pi}", [words[a], words[b]], split,
                                  frozenset({target_id}), {target_id: 1}))
    return queries

In [ ]:
def _show_corpus_and_first_stage() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    from collections import Counter
    print(f"corpus: {len(docs)} documents ({N_TOPICS} topics x 5 + {N_NOISE_DOCS} noise)")
    print("queries per split:", dict(Counter(q.split for q in queries)))
    bm25_top1_is_target = 0
    hybrid_recall_at_pool = evaluate_retriever(
        lambda toks: hybrid_rank(toks, index, dense_index), queries, POOL_N)
    for q in queries:
        top = bm25_rank(q.tokens, index)
        if top and top[0][0] in q.relevant_ids:
            bm25_top1_is_target += 1
    mean_recall, lo, hi = bootstrap_ci(hybrid_recall_at_pool.recall_at_k, seed=SEED)
    print(f"BM25's own top result is the target for {bm25_top1_is_target} of {len(queries)} "
          f"queries — length normalisation lets a shorter 3-word near-miss outscore the "
          f"4-word target that also matches both query words.")
    print(f"the first stage's own recall@{POOL_N} (is the target in the candidate pool at "
          f"all?): {mean_recall:.3f} [{lo:.3f},{hi:.3f}] — high: this is what 'tuned for "
          f"recall' means. Getting it near the TOP of that pool is this lesson's job.")


_show_corpus_and_first_stage()

## 3. Exercise 1 — two features BM25 and dense cosine cannot give you

`query_term_coverage` counts how many of the query's DISTINCT words a document contains at
all. `term_proximity` measures how close together, in TOKENS, the query's words sit inside a
document: the smallest window containing at least one occurrence of every matched query word,
expressed as (matched words) / (window length), 1.0 when they sit side by side.

Neither singles the target out on this corpus by itself. Coverage ties it with the two
near-misses that also carry both query words; proximity ties it with them too, or — for the
query that wraps from the fourth core word back to the first — favours those near-misses,
whose missing word leaves the pair closer together. What the two add is a way to tell those
on-topic documents from the rest of the pool; section 8 measures what they are worth combined
with the first stage's own scores and ranks.

<details><summary>💡 Hint 1 — what to think about</summary>

For proximity: if a document matches only ONE of the query's distinct words (or none), is
there a "closest together" to measure? Decide what that case returns before writing the
window search. For coverage: what should an EMPTY query cover?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`query_term_coverage`: distinct query words that are also in the document's own distinct
words, divided by the number of distinct query words; guard the empty-query case by
returning 0.0 rather than dividing by zero. `term_proximity`: if fewer than 2 distinct query
words are matched anywhere in the document, return 0.0. Otherwise walk the document once,
keeping only the positions of matched words, and slide a window over THAT filtered list
(classic minimum-window-covering-every-label two-pointer sweep): grow the right edge until
every matched word is inside the window, then shrink the left edge while it still is,
recording the smallest (position-of-right − position-of-left + 1) seen. Divide the number of
matched words by the smallest window found.
</details>

In [ ]:
def query_term_coverage(query_tokens: Sequence[str], doc_tokens: Sequence[str]) -> float:
    """The fraction of `query_tokens`'s DISTINCT words that appear anywhere in `doc_tokens`.

    Requirements, each of which is graded:
      * `len(set(query_tokens) & set(doc_tokens)) / len(set(query_tokens))`.
      * an EMPTY `query_tokens` returns `0.0`, never raises or divides by zero.

    Example:
        >>> query_term_coverage(["pump", "valve"], ["pump", "note", "gauge"])
        0.5
        >>> query_term_coverage(["pump", "valve"], ["pump", "valve", "valve"])
        1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def term_proximity(query_tokens: Sequence[str], doc_tokens: Sequence[str]) -> float:
    """How close together `query_tokens`'s matched, distinct words sit in `doc_tokens`.

    Requirements, each of which is graded:
      * fewer than 2 DISTINCT query words present anywhere in `doc_tokens` returns `0.0`
        (there is nothing to measure "closeness" between).
      * otherwise: find the smallest window `[left, right]` of POSITIONS in `doc_tokens` (both
        inclusive) that contains at least one occurrence of every matched distinct word, and
        return `(number of matched distinct words) / (right - left + 1)`.

    Example:
        >>> term_proximity(["pump", "valve"], ["pump", "note", "valve"])
        0.6666666666666666
        >>> term_proximity(["pump", "valve"], ["pump", "valve"])
        1.0
        >>> term_proximity(["pump", "zzz"], ["pump", "valve"])
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_features() -> None:
    assert query_term_coverage([], ["a", "b"]) == 0.0, "an empty query must cover 0.0, not raise"
    cov = query_term_coverage(["pump", "valve"], ["pump", "note", "gauge"])
    assert abs(cov - 0.5) < 1e-9, f"1 of 2 distinct query words present -> 0.5, got {cov}"
    cov_dupe = query_term_coverage(["pump", "valve"], ["pump", "valve", "valve", "valve"])
    assert abs(cov_dupe - 1.0) < 1e-9, (
        f"both distinct query words present -> 1.0 regardless of repeats, got {cov_dupe}"
    )

    assert term_proximity(["pump"], ["pump", "valve"]) == 0.0, (
        "only ONE distinct query word can be matched — there is nothing to be 'close' to"
    )
    assert term_proximity(["pump", "zzz"], ["pump", "valve"]) == 0.0, (
        "'zzz' never matches, so only one distinct word is matched — must still be 0.0"
    )
    adjacent = term_proximity(["pump", "valve"], ["pump", "valve"])
    assert abs(adjacent - 1.0) < 1e-9, f"both words adjacent -> window 2, 2/2=1.0, got {adjacent}"
    one_between = term_proximity(["pump", "valve"], ["pump", "note", "valve"])
    assert abs(one_between - 2 / 3) < 1e-9, (
        f"window is 3 tokens wide (positions 0..2) -> 2/3, got {one_between}"
    )
    farther = term_proximity(["pump", "valve"], ["pump", "note", "note", "note", "valve"])
    assert one_between > farther, (
        "a WIDER window between the same two matched words must score LOWER, not higher"
    )
    repeats = term_proximity(["pump", "valve"], ["valve", "x", "x", "x", "pump", "valve"])
    assert abs(repeats - 2 / 2) < 1e-9, (
        f"the window must be the SMALLEST that covers both words (positions 3-4: 'pump "
        f"valve'), not the first occurrence of each — expected 1.0, got {repeats}"
    )
    print("exercise 1 looks right — query_term_coverage and term_proximity agree with "
          "hand-counted windows, including the fewer-than-2-matched-words edge case")

In [ ]:
_try("exercise 1", _check_features)

## 4. Exercise 2 — assembling the reranker's feature matrix

One row per candidate, six columns, in this fixed order: the candidate's own BM25 score and
dense cosine (recomputed directly, not read off a truncated top-`POOL_N` list — a candidate
can be in the pool by ONE retriever's signal and still deserve the OTHER retriever's real
score, even a low one), each one's RECIPROCAL rank (0.0 if a retriever never ranked it at
all — BM25 drops zero-score documents, so "no rank" is common and must not become a division
by zero), and this lesson's own coverage and proximity.

<details><summary>💡 Hint — the approach, in words</summary>

Rank the whole corpus ONCE with each retriever, before you visit any candidate — never once
per candidate. From each ranking build two lookups: a document's score, and its position
counted from one. BM25's lookups can simply miss a candidate (its score was exactly zero, so it
was dropped): a missing candidate gets zero for both its score and its rank-reciprocal. Dense
never drops a document, so its lookups always have every candidate. Tokenise each candidate's
own text once, and hand those tokens to BOTH of exercise 1's functions.
</details>

In [ ]:
FEATURE_NAMES: tuple[str, ...] = ("bm25_score", "dense_cosine", "bm25_rank_recip",
                                   "dense_rank_recip", "coverage", "proximity")


def build_feature_matrix(query_tokens: Sequence[str], candidate_ids: Sequence[int],
                          index: Index, dense_index: DenseIndex, docs: Sequence[str]
                          ) -> tuple[np.ndarray, tuple[str, ...]]:
    """The reranking feature matrix for `candidate_ids`, row `i` for `candidate_ids[i]`.

    Requirements, each of which is graded:
      * columns, in this exact order, named by `FEATURE_NAMES`: `bm25_score`, `dense_cosine`,
        `bm25_rank_recip`, `dense_rank_recip`, `coverage`, `proximity`.
      * `bm25_score` is `bm25_score(query_tokens, doc_id, index)` — the real value, which may
        be `0.0` for a candidate BM25 never scored above zero.
      * `dense_cosine` is the candidate's own score from `dense_rank(query_tokens, dense_index)`
        (dense ranks every document, never drops one).
      * `bm25_rank_recip` is `1 / (1-based rank in bm25_rank(...))`, or `0.0` if the candidate
        is absent from that ranking; `dense_rank_recip` is the same rule for `dense_rank`.
      * `coverage`, `proximity`: exercise 1's two functions, called with `query_tokens` and
        `tokenize(docs[doc_id])`.
      * row order matches `candidate_ids` exactly; returns `(X, FEATURE_NAMES)` with
        `X.shape == (len(candidate_ids), 6)`.

    Example:
        >>> docs = ["case-1 pump valve", "gauge only"]
        >>> idx = build_index(docs)
        >>> di = build_dense_index(idx, k=2)
        >>> X, names = build_feature_matrix(["pump"], [0, 1], idx, di, docs)
        >>> X.shape
        (2, 6)
        >>> names[0]
        'bm25_score'
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_build_feature_matrix() -> None:
    docs = ["case-1 pump valve widget", "case-2 pump widget", "note only decoy words"]
    idx = build_index(docs)
    di = build_dense_index(idx, k=3)
    X, names = build_feature_matrix(["pump", "valve"], [0, 1, 2], idx, di, docs)
    assert names == FEATURE_NAMES, f"feature_names must be FEATURE_NAMES itself, got {names}"
    assert X.shape == (3, 6), f"expected shape (3, 6), got {X.shape}"

    bm25_col, dense_col, bm25_r_col, dense_r_col, cov_col, prox_col = range(6)
    assert X[0, bm25_col] == bm25_score(["pump", "valve"], 0, idx), (
        "column 0 must be the candidate's REAL bm25_score, not a value read off a ranked list"
    )
    assert X[2, bm25_col] == 0.0 and X[2, bm25_r_col] == 0.0, (
        "document 2 shares no term with the query — bm25_score is 0.0 AND bm25_rank_recip "
        "must be 0.0 (absent from bm25_rank's own list), not a crash from a missing key"
    )
    assert X[2, dense_r_col] != 0.0, (
        "dense_rank never drops a document — every candidate has a real, nonzero-rank dense "
        "reciprocal, even one BM25 dropped entirely"
    )
    top_bm25_id = bm25_rank(["pump", "valve"], idx)[0][0]
    top_row = [0, 1, 2].index(top_bm25_id)
    assert abs(X[top_row, bm25_r_col] - 1.0) < 1e-9, (
        f"the top-ranked document's bm25_rank_recip must be 1/1 = 1.0, got "
        f"{X[top_row, bm25_r_col]}"
    )
    assert abs(X[0, cov_col] - 1.0) < 1e-9, "document 0 has both query words -> coverage 1.0"
    assert abs(X[1, cov_col] - 0.5) < 1e-9, (
        f"document 1 holds 'pump' but not 'valve' -> coverage 1 of 2 = 0.5, got {X[1, cov_col]}"
    )
    print("exercise 2 looks right — build_feature_matrix reads real scores and ranks for "
          "every candidate, including one a retriever dropped entirely")

In [ ]:
_try("exercise 2", _check_build_feature_matrix)

## 5. Exercise 3 — the pointwise reranker: `sigmoid` and `train_pointwise_logreg`

The pointwise approach treats every candidate as its OWN yes/no classification: "is this
document the answer?" — ignoring that candidates from the same query are competing with each
other. It is the simplest possible reranker: logistic regression, trained by full-batch
gradient descent on the six features above.

<details><summary>💡 Hint — the approach, in words</summary>

`sigmoid`: limit the input to a moderate range first (the docstring suggests one), and only
then exponentiate, so a very confident score never overflows. `train_pointwise_logreg`: start
every weight and the bias at zero. Each epoch, compute every row's predicted probability at
once; a row's error is its probability minus its label. The weight gradient is the features
weighted by those errors, AVERAGED over the rows, plus `l2` times the current weights; the bias
gradient is the average error alone, with no `l2` term. Step both against their gradients by
`lr`, `epochs` times.
</details>

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """The logistic function, elementwise, safe for large `|z|`.

    Requirements, each of which is graded:
      * `1 / (1 + exp(-z))`.
      * clip `z` into a safe range (e.g. `[-30, 30]`) before exponentiating, so a large
        confident score never raises or warns from `exp` overflow.

    Example:
        >>> float(sigmoid(np.array([0.0]))[0])
        0.5
        >>> bool(sigmoid(np.array([100.0]))[0] > 0.999)
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


def train_pointwise_logreg(X: np.ndarray, y: np.ndarray, lr: float = 0.5, epochs: int = 400,
                            l2: float = 0.02) -> tuple[np.ndarray, float]:
    """Fit logistic regression `(w, b)` on `(X, y)` by full-batch gradient descent.

    `y[i]` is `1.0` if candidate `i` is its query's relevant document, else `0.0` — every row
    is its own independent binary label (the POINTWISE approach; contrast with exercise 4).

    Requirements, each of which is graded:
      * `w` starts at `np.zeros(X.shape[1])`, `b` at `0.0`.
      * each of `epochs` steps: `p = sigmoid(X @ w + b)`; `w -= lr * (X.T @ (p - y) / n + l2 * w)`;
        `b -= lr * mean(p - y)` — L2 regularises `w` only.

    Example:
        >>> X = np.array([[1.0], [1.0], [-1.0], [-1.0]])
        >>> y = np.array([1.0, 1.0, 0.0, 0.0])
        >>> w, b = train_pointwise_logreg(X, y, lr=0.5, epochs=200, l2=0.0)
        >>> bool(w[0] > 0)
        True
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_pointwise() -> None:
    z0 = sigmoid(np.array([0.0]))
    assert abs(float(z0[0]) - 0.5) < 1e-12, f"sigmoid(0) must be exactly 0.5, got {z0[0]}"
    big = sigmoid(np.array([1000.0, -1000.0]))
    assert np.all(np.isfinite(big)), (
        "sigmoid must not overflow on a large |z| — clip z before np.exp(-z)"
    )
    assert abs(float(big[0]) - 1.0) < 1e-6 and abs(float(big[1])) < 1e-6, (
        f"sigmoid(+1000) must be ~1.0 and sigmoid(-1000) ~0.0, got {big.tolist()} — clip to a "
        f"range WIDE enough (e.g. [-30, 30]) that the tails still reach 0 and 1"
    )

    X = np.array([[1.0], [1.0], [-1.0], [-1.0]])
    y = np.array([1.0, 1.0, 0.0, 0.0])
    w, b = train_pointwise_logreg(X, y, lr=0.5, epochs=300, l2=0.0)
    assert w[0] > 0.5, (
        f"one positive feature, perfectly separable, 300 steps at lr=0.5 -> w should be "
        f"clearly positive, got {w[0]:.4f}. Check the gradient's sign (p - y, not y - p)"
    )
    w0, _ = train_pointwise_logreg(X, y, lr=0.5, epochs=300, l2=0.0)
    w_reg, _ = train_pointwise_logreg(X, y, lr=0.5, epochs=300, l2=1.0)
    assert abs(w_reg[0]) < abs(w0[0]), (
        f"a large l2 must SHRINK |w| relative to l2=0, got |w|={abs(w_reg[0]):.4f} vs "
        f"{abs(w0[0]):.4f} — add l2 * w to the weight gradient (never to the bias gradient)"
    )
    w_start = np.zeros(1)
    w_after_0_epochs, b_after_0_epochs = train_pointwise_logreg(X, y, epochs=0)
    assert np.array_equal(w_after_0_epochs, w_start) and b_after_0_epochs == 0.0, (
        "0 epochs must return the untouched zero start (w=0, b=0.0)"
    )
    print("exercise 3 looks right — sigmoid is overflow-safe and train_pointwise_logreg "
          "separates linearly separable data and shrinks under l2")

In [ ]:
_try("exercise 3", _check_pointwise)

## 6. Exercise 4 — the pairwise reranker: `train_pairwise_ranknet`

Pointwise scoring never compares two candidates from the SAME query to each other — exactly
the comparison a rerank needs. RankNet (Burges et al., 2005; see `claims.yaml`) trains
instead on PAIRS: for every pair `(i, j)` of one query's own candidates where `i` should rank
above `j`, it models `P(i ≻ j) = sigmoid(s_i - s_j)` with `s = X @ w` (no bias: `s_i - s_j`
does not depend on `b`, so this exercise returns `b = 0.0`), and minimises the cross-entropy
cost for the hard label "`i` is preferred": `C_ij = -log(sigmoid(s_i - s_j))`.

<details><summary>💡 Hint — the approach, in words</summary>

First build the pairs, one query at a time: within each distinct query id, pair EVERY
relevant row with EVERY non-relevant row of that same query — never across two query ids. A
query with no such pair contributes none; with no pairs anywhere, return the zero start
untouched. Then descend as in exercise 3, but on pair DIFFERENCES: each pair's feature
difference (preferred row minus the other) and the sigmoid of its score difference under the
current weights. The preferred row always comes first, so every pair's target is 1 and its
error is that sigmoid minus one. The gradient is the differences weighted by those errors,
averaged over the pairs, plus `l2` times the weights.
</details>

In [ ]:
def train_pairwise_ranknet(X: np.ndarray, y: np.ndarray, query_ids: Sequence[int],
                            lr: float = 0.5, epochs: int = 400,
                            l2: float = 0.02) -> tuple[np.ndarray, float]:
    """Fit a linear ranking function by RankNet's pairwise logistic loss (see `claims.yaml`),
    full-batch gradient descent, with pairs built ONLY within each query.

    Requirements, each of which is graded:
      * for every distinct value of `query_ids`, pair each of that query's `y == 1` rows
        against each of that SAME query's `y == 0` rows; a pair never mixes two different
        query ids.
      * `w` starts at zeros; `b` is always `0.0` (RankNet's pairwise score difference does not
        depend on it).
      * each of `epochs` steps, with `I, J` the pairs' row indices: `diff = (X[I]-X[J]) @ w`;
        `p = sigmoid(diff)`; `w -= lr * ((X[I]-X[J]).T @ (p - 1) / len(pairs) + l2 * w)`.
      * zero pairs anywhere (e.g. every row has the same label) returns `(zeros, 0.0)`
        untouched, never a division by zero.

    Example:
        >>> X = np.array([[1.0], [-1.0], [1.0], [-1.0]])
        >>> y = np.array([1.0, 0.0, 1.0, 0.0])
        >>> qids = [0, 0, 1, 1]
        >>> w, b = train_pairwise_ranknet(X, y, qids, lr=0.5, epochs=200, l2=0.0)
        >>> bool(w[0] > 0)
        True
        >>> b
        0.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_pairwise() -> None:
    X = np.array([[1.0], [-1.0], [1.0], [-1.0]])
    y = np.array([1.0, 0.0, 1.0, 0.0])
    qids = [0, 0, 1, 1]
    w, b = train_pairwise_ranknet(X, y, qids, lr=0.5, epochs=300, l2=0.0)
    assert b == 0.0, f"RankNet's pairwise score difference cannot see b — must return 0.0, got {b}"
    assert w[0] > 0.5, (
        f"the positive feature always belongs to the preferred row in both queries — w should "
        f"be clearly positive, got {w[0]:.4f}"
    )

    no_pairs_X = np.array([[1.0], [2.0]])
    no_pairs_y = np.array([1.0, 1.0])   # both "relevant" -> no (relevant, non-relevant) pair
    w0, b0 = train_pairwise_ranknet(no_pairs_X, no_pairs_y, [0, 0], epochs=50)
    assert np.array_equal(w0, np.zeros(1)) and b0 == 0.0, (
        "a query with no (relevant, non-relevant) pair contributes nothing — with EVERY row "
        "the same label and no other query, the untouched zero start must come back unchanged"
    )

    cross_query_X = np.array([[1.0], [-1.0]])
    cross_query_y = np.array([1.0, 0.0])
    w_one_q, _ = train_pairwise_ranknet(cross_query_X, cross_query_y, [0, 0], epochs=300, l2=0.0)
    w_two_q, _ = train_pairwise_ranknet(cross_query_X, cross_query_y, [0, 1], epochs=300, l2=0.0)
    assert np.array_equal(w_two_q, np.zeros(1)), (
        "row 0 (query 0) and row 1 (query 1) are in DIFFERENT queries and must never be paired "
        "against each other — with no query contributing its OWN (relevant, non-relevant) "
        "pair, the result must be the untouched zero start, not the single-query answer "
        f"{w_one_q.round(4).tolist()}"
    )

    w_reg, _ = train_pairwise_ranknet(X, y, qids, lr=0.5, epochs=300, l2=1.0)
    w_noreg, _ = train_pairwise_ranknet(X, y, qids, lr=0.5, epochs=300, l2=0.0)
    assert abs(w_reg[0]) < abs(w_noreg[0]), (
        "a large l2 must shrink |w| here too — add l2 * w to the pairwise gradient"
    )
    print("exercise 4 looks right — train_pairwise_ranknet builds pairs only within a query, "
          "returns b=0.0, and shrinks under l2")

In [ ]:
_try("exercise 4", _check_pairwise)

## 7. Exercise 5 — applying a trained model: `rerank`

A trained model (exercise 3 or 4) scores candidates; `rerank` turns those scores into a new
order. This is the whole second stage: the first `depth` candidates of the first-stage
ranking get reordered by this; everything past `depth` keeps its first-stage position
untouched (section 8 measures what `depth` is worth).

<details><summary>💡 Hint — the approach, in words</summary>

`candidate_ids[i]` is scored `scores[i]`: the two sequences are paired by position, not two
lists to sort independently. Put the POSITIONS in order of score, highest first, breaking an
exact tie by the smaller candidate id — the tie-break every ranking function in this track
uses — then read the candidate ids off in that order.
</details>

In [ ]:
def rerank(candidate_ids: Sequence[int], scores: Sequence[float]) -> list[int]:
    """Reorder `candidate_ids` by `scores`, best first (`candidate_ids[i]` scored `scores[i]`).

    Requirements, each of which is graded:
      * sort by score descending; ties break by candidate id ascending.
      * every id in `candidate_ids` appears exactly once in the result, in the new order.

    Example:
        >>> rerank([7, 3, 9], [0.1, 0.9, 0.9])
        [3, 9, 7]
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_rerank() -> None:
    assert rerank([7, 3, 9], [0.1, 0.9, 0.9]) == [3, 9, 7], (
        "sort by score descending; an exact tie (3 and 9, both 0.9) breaks by ascending id"
    )
    assert rerank([1, 2, 3], [5.0, 5.0, 5.0]) == [1, 2, 3], (
        "every score tied -> pure ascending-id order"
    )
    assert rerank([], []) == [], "an empty pool reranks to an empty list"
    assert set(rerank([5, 1, 2], [0.0, -1.0, 3.0])) == {5, 1, 2}, (
        "every original id must still be present — rerank REORDERS, it never drops one"
    )
    print("exercise 5 looks right — rerank sorts by score descending with an ascending-id "
          "tie-break, and never drops or duplicates a candidate")

In [ ]:
_try("exercise 5", _check_rerank)

Given, not graded: a trained model bundled with the normalisation it was trained under
(`RerankerModel`), and `mean_reranked_ndcg`, which reranks each query's top `depth`
first-stage candidates and averages nDCG@10 — a thin composition of `build_feature_matrix`
and `rerank` (exercises 2 and 5) plus `ndcg_at_k` (T06-L01).

In [ ]:
class RerankerModel(NamedTuple):
    w: np.ndarray
    b: float
    mu: np.ndarray          # per-feature mean, from TRAIN candidates only
    sigma: np.ndarray       # per-feature std, from TRAIN candidates only (0 guarded to 1)
    feature_names: tuple[str, ...]


def mean_reranked_ndcg(model: RerankerModel, queries: Sequence[Query], index: Index,
                        dense_index: DenseIndex, docs: Sequence[str], depth: int = EVAL_K,
                        extra_feature: Callable[[int], float] | None = None) -> float:
    """Mean nDCG@EVAL_K after reranking each query's top `depth` first-stage candidates with
    `model`. `extra_feature`, if given, appends one more column (a doc id -> float lookup) —
    used only by exercise 6's demo, to score a model trained on a 7th feature."""
    scores = []
    for q in queries:
        base_order = [doc_id for doc_id, _ in hybrid_rank(q.tokens, index, dense_index)]
        head, tail = base_order[:depth], base_order[depth:]
        X, _ = build_feature_matrix(q.tokens, head, index, dense_index, docs)
        if extra_feature is not None:
            X = np.hstack([X, np.array([[extra_feature(doc_id)] for doc_id in head])])
        X_norm = (X - model.mu) / model.sigma
        raw_scores = (X_norm @ model.w + model.b).tolist()
        final = rerank(head, raw_scores) + tail
        scores.append(ndcg_at_k(final, q.relevance, EVAL_K))
    return float(np.mean(scores)) if scores else 0.0


def _fit_reranker(queries: Sequence[Query], index: Index, dense_index: DenseIndex,
                   docs: Sequence[str], trainer: str, l2: float = 0.02, lr: float = 0.5,
                   epochs: int = 400,
                   extra_feature: Callable[[int], float] | None = None) -> RerankerModel:
    """Given: build (X, y) over every TRAIN query's own first-stage candidates, normalise by
    TRAIN's own mean/std, and fit `trainer` ("pointwise" or "pairwise")."""
    X_parts, y_parts, qid_parts = [], [], []
    for qi, q in enumerate(queries):
        ids = first_stage_candidates(q.tokens, index, dense_index)
        X, names = build_feature_matrix(q.tokens, ids, index, dense_index, docs)
        if extra_feature is not None:
            X = np.hstack([X, np.array([[extra_feature(doc_id)] for doc_id in ids])])
            names = names + ("recency_score",)
        y = np.array([1.0 if doc_id in q.relevant_ids else 0.0 for doc_id in ids])
        X_parts.append(X)
        y_parts.append(y)
        qid_parts += [qi] * len(ids)
    X_all = np.vstack(X_parts)
    y_all = np.concatenate(y_parts)
    mu = X_all.mean(axis=0)
    sigma = X_all.std(axis=0)
    sigma = np.where(sigma == 0, 1.0, sigma)
    X_norm = (X_all - mu) / sigma
    if trainer == "pointwise":
        w, b = train_pointwise_logreg(X_norm, y_all, lr=lr, epochs=epochs, l2=l2)
    else:
        w, b = train_pairwise_ranknet(X_norm, y_all, qid_parts, lr=lr, epochs=epochs, l2=l2)
    return RerankerModel(w, b, mu, sigma, names)

## 8. The gain, measured with an interval — and the depth curve

Everything below is given: composition of the exercises above, run once, here, on `val`
queries only. `test` is not touched until section 9's single, graded report — see why there.

Two cautions the numbers cannot print for themselves. The four queries of one topic share its
target and its near-misses, so they are not independent draws: an interval that resamples
queries one at a time is likely narrower than one that resampled whole topics would be. And
the cost is timed per stage — the first stage alone, then everything the reranker adds —
because one total would hide which of the two the rerank depth actually changes.

In [ ]:
def run_gains_and_depth_curve() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    train_q = [q for q in queries if q.split == "train"]
    val_q = [q for q in queries if q.split == "val"]

    pointwise = _fit_reranker(train_q, index, dense_index, docs, "pointwise")
    pairwise = _fit_reranker(train_q, index, dense_index, docs, "pairwise")

    def baseline_ndcg_and_mrr(qs):
        ndcgs, mrrs = [], []
        for q in qs:
            ranked = [doc_id for doc_id, _ in hybrid_rank(q.tokens, index, dense_index)]
            ndcgs.append(ndcg_at_k(ranked, q.relevance, EVAL_K))
            mrrs.append(reciprocal_rank(ranked, q.relevant_ids))
        return ndcgs, mrrs

    def reranked_ndcg_and_mrr(model):
        ndcgs, mrrs = [], []
        for q in val_q:
            base_order = [doc_id for doc_id, _ in hybrid_rank(q.tokens, index, dense_index)]
            head, tail = base_order[:EVAL_K], base_order[EVAL_K:]
            X, _ = build_feature_matrix(q.tokens, head, index, dense_index, docs)
            scores = (((X - model.mu) / model.sigma) @ model.w + model.b).tolist()
            final = rerank(head, scores) + tail
            ndcgs.append(ndcg_at_k(final, q.relevance, EVAL_K))
            mrrs.append(reciprocal_rank(final, q.relevant_ids))
        return ndcgs, mrrs

    base_ndcgs, base_mrrs = baseline_ndcg_and_mrr(val_q)
    print(f"first stage alone (val, n={len(val_q)}): "
          f"nDCG@{EVAL_K}={np.mean(base_ndcgs):.3f}  MRR={np.mean(base_mrrs):.3f}")
    for name, model in (("pointwise", pointwise), ("pairwise", pairwise)):
        ndcgs, mrrs = reranked_ndcg_and_mrr(model)
        d_ndcg, lo_n, hi_n = paired_bootstrap_diff(base_ndcgs, ndcgs, seed=SEED)
        d_mrr, lo_m, hi_m = paired_bootstrap_diff(base_mrrs, mrrs, seed=SEED)
        print(f"{name:9s} reranked (depth={EVAL_K}): nDCG@{EVAL_K}={np.mean(ndcgs):.3f}  "
              f"MRR={np.mean(mrrs):.3f}   gain vs first stage: "
              f"ΔnDCG={d_ndcg:+.3f} [{lo_n:+.3f},{hi_n:+.3f}]  "
              f"ΔMRR={d_mrr:+.3f} [{lo_m:+.3f},{hi_m:+.3f}]")

    print(f"\nrerank-depth curve (pairwise, val, n={len(val_q)}) — quality against how many "
          f"first-stage candidates get reordered, and what each stage costs:")
    depth_ndcg: dict[int, float] = {}
    for depth in (1, 2, 3, 5, 10, 20):
        first_s = second_s = 0.0
        ndcgs = []
        for q in val_q:
            t0 = time.perf_counter()
            base_order = [doc_id for doc_id, _ in hybrid_rank(q.tokens, index, dense_index)]
            t1 = time.perf_counter()
            head, tail = base_order[:depth], base_order[depth:]
            X, _ = build_feature_matrix(q.tokens, head, index, dense_index, docs)
            scores = (((X - pairwise.mu) / pairwise.sigma) @ pairwise.w + pairwise.b).tolist()
            final = rerank(head, scores) + tail
            t2 = time.perf_counter()
            first_s += t1 - t0
            second_s += t2 - t1
            ndcgs.append(ndcg_at_k(final, q.relevance, EVAL_K))
        depth_ndcg[depth] = float(np.mean(ndcgs))
        print(f"  depth={depth:2d}  nDCG@{EVAL_K}={depth_ndcg[depth]:.3f}   first stage "
              f"≈{first_s / len(val_q) * 1000:.3f} ms/query, rerank "
              f"≈{second_s / len(val_q) * 1000:.3f} ms/query ({first_s + second_s:.4f} s "
              f"measured over {len(val_q)} queries)")
    best = max(depth_ndcg.values())
    plateau = min(depth for depth, value in depth_ndcg.items() if value == best)
    if plateau == max(depth_ndcg):
        print(f"\nnDCG@{EVAL_K} was still climbing at the deepest depth tried, {plateau}.")
    else:
        print(f"\nnDCG@{EVAL_K} stops climbing at depth={plateau}: reranking any deeper bought "
              f"nothing on val.")
    print("wall time, and anything computed from it (marked ≈), is measured on THIS machine and "
          "will differ on yours: compare the columns, not the milliseconds. The rerank column is "
          "the whole second stage, including the two corpus-wide rankings build_feature_matrix "
          "reads its ranks from once per query, so only part of it grows with depth; a reranker "
          "that costs more per candidate makes that column climb with depth, and the depth where "
          "nDCG stops climbing is then the most it is worth paying for.")


_try("gains and depth curve", run_gains_and_depth_curve,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 9. Exercise 6 — the trap: choosing a model by peeking at test

Every choice — which trained model to ship, which feature to keep — is made from `val` (look
at it as much as you like); `test` is read exactly ONCE, to report the number you actually
stand behind. `select_and_report` is graded against plain `scorers` callables, not real
models, so it is graded on its OWN mistake, independent of exercises 2-5.

<details><summary>💡 Hint — the approach, in words</summary>

Score every model on `val_queries` first, and only on them. Pick the name with the largest
val score, keeping the first name in the mapping's own order on an exact tie (Python's built-in
maximum already behaves that way). Only then score that ONE chosen model on `test_queries`,
exactly once — never any other model, and never before the choice is made — and return its
name with that number.
</details>

In [ ]:
def select_and_report(scorers: Mapping[str, Callable[[Sequence["Query"]], float]],
                       val_queries: Sequence["Query"],
                       test_queries: Sequence["Query"]) -> tuple[str, float]:
    """Choose the better of `scorers` by `val_queries`, then report ONLY its `test_queries` score.

    `scorers[name](queries)` returns that model's mean metric over `queries` — read the whole
    sequence to do so (this is what a `TripwireQueries` instance measures).

    Requirements, each of which is graded:
      * every name in `scorers` is scored on `val_queries`; the NAME with the strictly larger
        value is "best" (an exact tie keeps the first name in `scorers`' own iteration order).
      * `test_queries` is read for scoring EXACTLY ONE model — the chosen one — and only AFTER
        that choice is already made from `val_queries` alone.
      * returns `(best_name, scorers[best_name](test_queries))`.

    Example:
        >>> scorers = {"a": lambda qs: 0.9 if qs == "val" else 0.5,
        ...            "b": lambda qs: 0.2 if qs == "val" else 0.9}
        >>> select_and_report(scorers, "val", "test")
        ('a', 0.5)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
class TripwireQueries:
    """Wraps a sequence of queries so reading it can be COUNTED. Reading means iterating it
    (`for q in ...`) or wrapping it in `list(...)` — what a real evaluator does to compute a
    mean over it. `touch_count` is how many separate reads have happened; a grader (or you)
    can use it to check something never claimed in prose: whether test data was actually read
    during model selection. Only ever used on data you are deliberately watching — never
    write anything like it into a real held-out set."""

    def __init__(self, queries: Sequence["Query"]):
        self._queries = list(queries)
        self.touch_count = 0

    def __iter__(self):
        self.touch_count += 1
        return iter(self._queries)

    def __len__(self) -> int:
        return len(self._queries)


def _fixed_scorer(val_marker, val_value: float, test_marker, test_value: float):
    """A scorer whose return value depends only on WHICH marker object it was called with —
    calling it also READS its argument (like a real evaluator), so a TripwireQueries wrapper
    correctly measures whether it was called."""
    def scorer(queries):
        list(queries)   # read every element, exactly as a real "compute the mean" would
        if queries is val_marker:
            return val_value
        if queries is test_marker:
            return test_value
        raise AssertionError("scorer called with neither the val nor the test marker")
    return scorer


def _check_select_and_report() -> None:
    val = [0, 1, 2]

    # The exact shape of this lesson's trap: "with_noise" looks worse on val (honestly worse)
    # but better on test — a correct implementation must still pick by val.
    test_a = TripwireQueries([10, 11])
    result = select_and_report(
        {"no_noise": _fixed_scorer(val, 0.90, test_a, 0.75),
         "with_noise": _fixed_scorer(val, 0.80, test_a, 0.95)},
        val, test_a)
    assert result == ("no_noise", 0.75), (
        f"val prefers 'no_noise' (0.90 > 0.80); the choice must follow val, not test (where "
        f"'with_noise' looks better, 0.95 > 0.75) — got {result}"
    )
    assert test_a.touch_count == 1, (
        f"test_queries was read {test_a.touch_count} time(s) — it must be read EXACTLY ONCE, "
        f"to report the chosen model's own number, never during selection"
    )

    # argmax, not "always the first key": the best model here is NOT first in the mapping.
    test_b = TripwireQueries(["x"])
    result2 = select_and_report(
        {"first": _fixed_scorer(val, 0.10, test_b, 0.99),
         "second": _fixed_scorer(val, 0.95, test_b, 0.40),
         "third": _fixed_scorer(val, 0.50, test_b, 0.10)},
        val, test_b)
    assert result2 == ("second", 0.40), (
        f"'second' has the highest VAL score (0.95) even though it is not first in the "
        f"mapping and does not have the highest TEST score — got {result2}"
    )
    assert test_b.touch_count == 1, (
        f"test_queries was read {test_b.touch_count} time(s) scoring 3 candidate models — "
        f"only the ONE chosen model may ever be scored on test"
    )

    # An exact tie on val keeps the FIRST name, in the mapping's own order.
    test_c = TripwireQueries(["y"])
    result3 = select_and_report(
        {"alpha": _fixed_scorer(val, 0.5, test_c, 0.1),
         "beta": _fixed_scorer(val, 0.5, test_c, 0.9)},
        val, test_c)
    assert result3[0] == "alpha", (
        f"'alpha' and 'beta' tie exactly on val (0.5 == 0.5) — the tie must keep 'alpha', the "
        f"first name in the mapping, got {result3[0]!r}"
    )
    print("exercise 6 looks right — select_and_report picks by val's own argmax (not test's, "
          "and not just the first key), and reads test_queries exactly once")

In [ ]:
_try("exercise 6", _check_select_and_report)

## 10. Common mistakes

- **Coverage or proximity computed from the wrong tokens.** Both exercise 1 functions take
  the CANDIDATE document's own tokens, not the whole corpus's vocabulary or the query's.
- **`term_proximity` on the first occurrence of each word, not the closest.** The smallest
  window can start later in the document than a word's first appearance; the sliding-window
  sweep in exercise 1's hint finds it, a "first occurrence of each" shortcut does not.
- **Looking a candidate's BM25 score up in a truncated ranked list instead of computing it.**
  `bm25_rank(...)`'s own list is cut at whatever depth called it; a candidate the DENSE
  retriever contributed but BM25 ranked lower still has a real (possibly 0.0) BM25 score —
  call `bm25_score` directly, or read a `dict(bm25_rank(...))` built from the FULL ranking.
- **A missing rank silently becoming `1/0`.** BM25 drops zero-score documents; a candidate
  absent from its ranking has NO reciprocal rank to give — `0.0`, not a `ZeroDivisionError`
  and not a `KeyError`.
- **Training the pointwise and pairwise rerankers on raw, unnormalised features.** `bm25_score`
  and the two reciprocal ranks live on very different numeric scales, and a weight's gradient
  scales with its feature's values: without the z-score normalisation `_fit_reranker` applies,
  one learning rate is too large for the biggest-valued feature and too small for the rest.
- **Pairing RankNet's candidates across two different queries.** Only documents retrieved for
  the SAME query are ever compared against each other (see `claims.yaml`); a pair drawn from
  two different queries compares things that were never in competition.
- **Choosing a model, or a feature, by its score on `test_queries`.** Exercise 6's whole
  point: `val_queries` is for every choice you make; `test_queries` is read exactly once, to
  report the number for the choice you already made.
- **Reading a bigger TEST number as proof a change helped.** Choosing the best of several
  models BY their test scores makes the reported number the largest of several noisy
  estimates, so it is biased upward. The demo just below measures by how much, against a
  fresh split that no selection step ever reads.

Given, not graded: this exercise's own demonstration, with REAL trained rerankers. A
`recency_score` feature — "how recently was this report filed" — is a plausible thing a
stakeholder might ask for; this corpus has no real timestamps, so it is simulated as a random
per-document tag, unrelated to relevance by construction. Each trial trains `N_NOISE_MODELS`
rivals to the plain reranker, each with its own tag. `select_and_report` chooses honestly,
from val; `_leaky_select_and_report` — never call this on real work — picks by peeking at
test. The leaked pick's test number can never be below the honest pick's (it is the maximum),
which is exactly why it cannot be trusted; the fresh split, which no selection step reads,
referees. One trial can go either way by luck, so the demo runs `N_TRIALS` and averages.

In [ ]:
def _leaky_select_and_report(scorers, test_queries):
    """What NOT to do: choose by TEST performance instead of val. Given so you can watch the
    damage happen — never called on anything that matters."""
    test_scores = {name: fn(test_queries) for name, fn in scorers.items()}
    best_name = max(test_scores, key=test_scores.get)
    return best_name, test_scores[best_name]


def run_leakage_trap_demo() -> None:
    docs, doc_meta = build_corpus()
    index = build_index(docs)
    dense_index = build_dense_index(index, DENSE_K)
    queries = build_queries(doc_meta)
    split = {name: [q for q in queries if q.split == name]
             for name in ("train", "val", "test", "fresh")}

    def scorer_for(model, extra_feature):
        return lambda qs: mean_reranked_ndcg(model, qs, index, dense_index, docs,
                                              extra_feature=extra_feature)

    plain = _fit_reranker(split["train"], index, dense_index, docs, "pointwise")
    tags = np.random.default_rng(NOISE_SEED).normal(size=(N_TRIALS, N_NOISE_MODELS, len(docs)))
    print(f"{N_TRIALS} trials, each choosing among the plain reranker and {N_NOISE_MODELS} "
          f"'recency' rivals: honestly from {len(split['val'])} val queries, or by peeking at "
          f"{len(split['test'])} test queries; {len(split['fresh'])} fresh queries referee.\n")
    print(f"{'trial':>5}  {'honest pick':<11} {'leaked pick':<11}  test: honest  leaked"
          f"   fresh: honest  leaked   (nDCG@{EVAL_K})")
    apparent, real, n_differ, honest_reads, leaky_reads = [], [], 0, [], []
    for trial in range(N_TRIALS):
        scorers = {"plain": scorer_for(plain, None)}
        for k in range(N_NOISE_MODELS):
            def extra_feature(doc_id, tag=tags[trial, k]):
                return float(tag[doc_id])
            model = _fit_reranker(split["train"], index, dense_index, docs, "pointwise",
                                  extra_feature=extra_feature)
            scorers[f"recency{k + 1}"] = scorer_for(model, extra_feature)

        test_watch = TripwireQueries(split["test"])
        honest_name, honest_test = select_and_report(scorers, split["val"], test_watch)
        leaky_watch = TripwireQueries(split["test"])
        leaky_name, leaky_test = _leaky_select_and_report(scorers, leaky_watch)
        honest_fresh = scorers[honest_name](split["fresh"])
        leaky_fresh = scorers[leaky_name](split["fresh"])

        honest_reads.append(test_watch.touch_count)
        leaky_reads.append(leaky_watch.touch_count)
        n_differ += honest_name != leaky_name
        apparent.append(leaky_test - honest_test)
        real.append(leaky_fresh - honest_fresh)
        print(f"{trial + 1:>5}  {honest_name:<11} {leaky_name:<11}  {honest_test:>12.4f} "
              f"{leaky_test:>7.4f}   {honest_fresh:>13.4f} {leaky_fresh:>7.4f}")

    honest_desc = ("exactly once in every trial" if set(honest_reads) == {1}
                   else f"{honest_reads} times, trial by trial")
    leaky_desc = (f"{leaky_reads[0]} times in every trial" if len(set(leaky_reads)) == 1
                  else f"{leaky_reads} times, trial by trial")
    print(f"\ntripwire: the honest selection read test_queries {honest_desc}; the peeking one "
          f"read it {leaky_desc}, once per candidate model.")
    overstated = [a - r for a, r in zip(apparent, real)]
    mean_over, lo, hi = bootstrap_ci(overstated, seed=SEED)
    print(f"the two picks differed in {n_differ} of {N_TRIALS} trials. Averaged over all "
          f"{N_TRIALS}, the leaked pick looked {float(np.mean(apparent)):+.4f} nDCG@{EVAL_K} "
          f"better than the honest pick on test, the split it was chosen by, and was "
          f"{float(np.mean(real)):+.4f} better on fresh queries no selection step read.")
    print(f"peeking overstated its pick's advantage by {mean_over:+.4f} nDCG@{EVAL_K} per trial, "
          f"95% bootstrap interval over trials [{lo:+.4f}, {hi:+.4f}].")
    if lo > 0:
        print("the whole interval sits above zero: the overstatement comes from choosing by "
              "test, not from the luck of one trial.")
    elif mean_over > 0:
        print("the interval still reaches zero: with this few trials the overstatement is "
              "suggestive, not established — raise N_TRIALS and re-run.")
    else:
        print("no overstatement on average in these trials: the leaked picks' test lead held up "
              "on fresh queries this time — raise N_TRIALS and re-run before trusting that.")

_try("leakage trap demo", run_leakage_trap_demo,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 5", "exercise 6"))

## 11. Self-check

1. A candidate is in the pool because DENSE ranked it well, but it shares no token with the
   query at all. Its `bm25_rank_recip` feature should be:
   - (a) `1.0`, since it is a valid first-stage candidate
   - (b) undefined — `build_feature_matrix` should raise for this candidate
   - (c) `0.0`

2. Two candidates from the SAME query: one relevant, one not. Under the POINTWISE reranker,
   how are they related during training?
   - (a) they form one training example each, with independent binary labels — the training
     objective never directly compares their two scores to each other
   - (b) they always form exactly one RankNet pair
   - (c) whichever was retrieved first is treated as the label for both

3. `term_proximity(["pump", "valve"], doc_tokens)` returns a HIGHER number for document A
   than document B. What does that mean on its own, without also knowing `coverage`?
   - (a) document A mentions "pump" and "valve" more often, in total, than document B
   - (b) in document A, the CLOSEST "pump" is to a "valve" is a smaller window than in
     document B, among the words that ARE present — it says nothing about how OFTEN
   - (c) document A is more relevant than document B

4. Your `select_and_report` is handed a val-best model and, separately, a model that scores
   higher on `test_queries` but was never preferred by val. The correct number to report is:
   - (a) whichever model scores higher on test — that is what test is for
   - (b) the mean of both models' test scores
   - (c) the val-chosen model's OWN test score, even if a different model would have scored
     higher there

5. A `TripwireQueries`-wrapped test split shows `touch_count == 2` after your model-selection
   function runs, for 2 candidate models. The most likely explanation is:
   - (a) the function scored BOTH candidates on test, which is how test leaks into a
     selection decision that is supposed to depend on val alone
   - (b) nothing is wrong — reading it twice is harmless bookkeeping
   - (c) the wrapper itself is buggy and double-counts a single read

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built

`RerankerModel` is the record shape; `build_feature_matrix` is the feature bridge from a
first-stage candidate to a row a model can score; `train_pointwise_logreg` and
`train_pairwise_ranknet` are the two rerankers; `rerank` applies either; `select_and_report`,
watched by a `TripwireQueries`, is the discipline that keeps a reported gain honest. A
reranker's number is only as trustworthy as the split it was measured on.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check, the exercises that check relies on)
        ("exercise 1", _check_features, ()),
        ("exercise 2", _check_build_feature_matrix, ()),
        ("exercise 3", _check_pointwise, ()),
        ("exercise 4", _check_pairwise, ()),
        ("exercise 5", _check_rerank, ()),
        ("exercise 6", _check_select_and_report, ()),
    )
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.